# Week 49, Day 5: Mock interview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w049d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
# Steps 2 and 3 for "is any value repeated within k positions of itself?"
def brute(nums, k):
    return any(nums[i] == nums[j] for i in range(len(nums)) for j in range(i + 1, min(len(nums), i + k + 1)))

# Step 4: the repeated work is re-scanning the last k values; a set of the window avoids it.
def better(nums, k):
    window = set()
    for i, x in enumerate(nums):
        if x in window:
            return True
        window.add(x)
        if i >= k:
            window.discard(nums[i - k])
    return False

for nums, k in [([1, 2, 3, 1], 3), ([1, 2, 3, 1, 2, 3], 2), ([], 1), ([7, 7], 1)]:
    print(nums, k, brute(nums, k), better(nums, k))

In [ ]:
def longest_run(nums):
    values = set(nums)
    best = 0
    for x in values:
        if x - 1 not in values:              # only start at the beginning of a run
            length = 1
            while x + length in values:
                length += 1
            best = max(best, length)
    return best

for case in [[100, 4, 200, 1, 3, 2], [0, 0, 1], [], [5], [-1, -2, 0, 9]]:
    print(case, longest_run(case))

In [ ]:
def stream_runs(values):
    length, best, out = {}, 0, []
    for x in values:
        if x not in length:
            left = length.get(x - 1, 0)        # the run ending just before x
            right = length.get(x + 1, 0)       # the run starting just after x
            total = left + right + 1
            length[x] = total
            length[x - left] = total           # update the new run's two ends
            length[x + right] = total
            best = max(best, total)
        out.append(best)
    return out

print(stream_runs([5, 1, 3, 2, 4, 9]))

> **Key idea.** In a problem-solving interview, the process is judged as well as the result. Follow seven steps out loud: understand (ask about sizes, edge cases, bad input), examples, brute force with its cost, improve by naming the repeated work and the pattern that removes it, code clearly, test with your examples and edge cases, and analyse time and space. Expect follow-ups that change a constraint (memory, streaming, repeated queries, extra space) and answer them from why your solution works. When stuck, go back to examples, simplify, run through the toolbox, or write the brute force.

### Exercise: Interview 1: consecutive runs

Write `longest_run(nums)` in O(n) average time with a set (no sorting), and `stream_runs(values)`: the longest run so far after each arriving value, in O(1) average per value with a dict of run endpoints. Then `runs(nums)`: every maximal run as `(start, end)` pairs, sorted by start, in O(n) average.

In [ ]:
def longest_run(nums):
    ___

def stream_runs(values):
    ___

def runs(nums):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_runs():\n    \"runs\"\n    assert longest_run([100, 4, 200, 1, 3, 2]) == 4 and longest_run([0, 0, 1]) == 2 and longest_run([]) == 0\n    assert stream_runs([5, 1, 3, 2, 4, 9]) == [1, 1, 1, 3, 5, 5] and stream_runs([2, 2]) == [1, 1]\n    assert runs([100, 4, 200, 1, 3, 2, 101]) == [(1, 4), (100, 101), (200, 200)] and runs([]) == []\n    rng = random.Random(1)\n    for _ in range(200):\n        xs = [rng.randint(-10, 10) for _ in range(rng.randint(0, 15))]\n        s = sorted(set(xs))\n        groups = []\n        for v in s:\n            if groups and groups[-1][1] == v - 1:\n                groups[-1][1] = v\n            else:\n                groups.append([v, v])\n        assert runs(xs) == [tuple(g) for g in groups]\n        assert longest_run(xs) == max((b - a + 1 for a, b in groups), default=0)\n        assert stream_runs(xs) == [longest_run(xs[:i + 1]) for i in range(len(xs))]\n    assert \"sort\" not in source.split(\"def longest_run\")[1].split(\"def stream_runs\")[0]")

### Exercise: Interview 2: inserting an interval

You're given a list of non-overlapping intervals `(start, end)`, sorted by start, and one new interval. Write `insert_interval(intervals, new)`: the list after adding the new interval and merging any it overlaps or touches, in O(n) (no sorting: the input is sorted already). Follow-up: `free_slots(busy, day_start, day_end)`: the gaps of at least 1 minute in a sorted, non-overlapping list of busy intervals, within the day.

In [ ]:
def insert_interval(intervals, new):
    ___

def free_slots(busy, day_start, day_end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_insert():\n    \"inserting\"\n    assert insert_interval([(1, 3), (6, 9)], (2, 5)) == [(1, 5), (6, 9)]\n    assert insert_interval([(1, 2), (3, 5), (6, 7), (8, 10), (12, 16)], (4, 8)) == [(1, 2), (3, 10), (12, 16)]\n    assert insert_interval([], (5, 7)) == [(5, 7)] and insert_interval([(1, 2)], (3, 4)) == [(1, 2), (3, 4)] and insert_interval([(1, 2)], (2, 4)) == [(1, 4)]\n    assert free_slots([(9, 10), (12, 13)], 8, 17) == [(8, 9), (10, 12), (13, 17)] and free_slots([(8, 17)], 8, 17) == []\n    rng = random.Random(2)\n    for _ in range(200):\n        pts = sorted(rng.sample(range(40), rng.randint(0, 6) * 2))\n        ivs = [(pts[i], pts[i + 1]) for i in range(0, len(pts), 2)]\n        s = rng.randint(0, 40)\n        new = (s, s + rng.randint(0, 8))\n        merged = []\n        for a, b in sorted(ivs + [new]):\n            if merged and a <= merged[-1][1]:\n                merged[-1] = (merged[-1][0], max(merged[-1][1], b))\n            else:\n                merged.append((a, b))\n        assert insert_interval(ivs, new) == merged\n    assert \"sort\" not in source.split(\"def insert_interval\")[1].split(\"def free_slots\")[0]")

### Exercise: Interview 3: decoding nested repeats

Write `decode(s)` for strings like `"3[a2[c]]"` → `"accaccacc"`: a number followed by brackets repeats the bracketed part; brackets nest; numbers may have several digits. Use a stack of `(string_so_far, repeat_count)` pairs, pushing when you see `[` and popping at `]`. Raise `ValueError` for unbalanced brackets or a `[` not preceded by a number. Follow-up: `decoded_length(s)`: the length of the decoded string **without** building it (it may be astronomically long), with the same stack holding lengths.

In [ ]:
def decode(s):
    ___

def decoded_length(s):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_decode():\n    \"decoding\"\n    assert decode(\"3[a2[c]]\") == \"accaccacc\" and decode(\"2[abc]3[cd]ef\") == \"abcabccdcdcdef\" and decode(\"10[x]\") == \"x\" * 10\n    assert decode(\"\") == \"\" and decode(\"ab\") == \"ab\" and decode(\"2[]\") == \"\"\n    for bad in [\"3[a\", \"a]\", \"[a]\", \"2[a]]\"]:\n        try:\n            decode(bad)\n            assert False, bad\n        except ValueError:\n            pass\n    assert decoded_length(\"3[a2[c]]\") == 9 and decoded_length(\"100[100[100[100[ab]]]]\") == 2 * 10**8\n    assert decoded_length(\"2[abc]3[cd]ef\") == 14")

### Exercise: Interview 4: the first common time

Two people each have a sorted list of free minutes (whole numbers, no repeats). Write `first_common(a, b)`: the earliest minute both are free, or `None`, with two pointers in O(len(a) + len(b)) and O(1) extra space (no sets: that's the follow-up the interviewer asked for). Then `all_common(lists)`: every minute free for **all** of k people, by repeatedly applying a two-list version.

In [ ]:
def first_common(a, b):
    ___

def all_common(lists):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_common():\n    \"common times\"\n    assert first_common([1, 4, 7, 9], [2, 3, 7, 9]) == 7 and first_common([1, 2], [3, 4]) is None and first_common([], [1]) is None\n    assert all_common([[1, 2, 5, 9], [2, 5, 6, 9], [0, 2, 9]]) == [2, 9] and all_common([]) == [] and all_common([[3, 4]]) == [3, 4]\n    rng = random.Random(3)\n    for _ in range(200):\n        ls = [sorted(rng.sample(range(30), rng.randint(0, 12))) for _ in range(rng.randint(1, 4))]\n        both = sorted(set(ls[0]) & set(ls[-1]))\n        assert first_common(ls[0], ls[-1]) == (both[0] if both else None)\n        assert all_common(ls) == sorted(set.intersection(*map(set, ls)))\n    assert \"set(\" not in source and \"in b\" not in source")

### Bug hunt: Testing your own code: the empty case

In a mock interview, this solution passed the interviewer's main example, then failed the first edge case they tried: an empty list crashes with a `ValueError` from `max`. Handle it so this prints `0 3`.

In [ ]:
def longest_increasing_run(nums):
    lengths = [1]
    for a, b in zip(nums, nums[1:]):
        lengths.append(lengths[-1] + 1 if b > a else 1)
    return max(lengths[:len(nums)])

print(longest_increasing_run([]), longest_increasing_run([1, 2, 3, 1, 2]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"edge cases\"\n    expect_output(\"0 3\")\n    assert longest_increasing_run([5]) == 1 and longest_increasing_run([3, 2, 1]) == 1")

### Bug hunt: Testing your own code: duplicates

This "longest run" counts runs from values whose predecessor is missing, but it loops over the **list** rather than the set, so with duplicates it counts the same run several times and, worse, a long run of repeated starts makes it quadratic. The answer is still right here, but the work isn't: this prints `4 12`, counting 12 steps. Loop over the set so it prints `4 4`.

In [ ]:
def longest_run(nums):
    values, best, steps = set(nums), 0, 0
    for x in nums:
        if x - 1 not in values:
            length = 1
            steps += 1
            while x + length in values:
                length += 1
                steps += 1
            best = max(best, length)
    return best, steps

print(*longest_run([1, 2, 3, 4, 1, 1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"once per value\"\n    expect_output(\"4 4\")")

### Puzzle: Inserting an interval

Arrange the lines into the O(n) interval insertion. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # longest_run([9, 1, 4, 7, 3, -1, 0, 5, 8, -1, 6])
q = ___   # decode("2[a3[b]]")
r = ___   # insert_interval([(1, 2), (5, 6)], (2, 5))
s = ___   # the number of the method's step where you state the brute force's cost (1 to 7)
t = ___   # the time cost of the set-based longest_run, like "O(n)"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 7 and q == \"abbbabbb\" and r == [(1, 6)], \"p, q and r\"\n    assert s == 3 and t.replace(\" \", \"\") == \"O(n)\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: the sliding median

An interviewer's final question: "Given a list of numbers and k, return the **median** of every window of size k. Now do it faster than sorting every window." Write `sliding_median(nums, k)` (as floats when k is even, the average of the two middle values; the value itself when k is odd) in O(n log k): keep two heaps as in Week 47 Day 4 (a max-heap `low` and a min-heap `high`, `low` holding the extra item when k is odd), and remove the outgoing value by **lazy deletion** (Week 47 Day 2): count it in a dict of values to delete, adjust the sizes you track by hand, and only actually pop a value when it reaches the top of its heap. The checks compare with sorting each window and time 100,000 numbers with k = 1,000.

In [ ]:
import heapq

def sliding_median(nums, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, time, statistics\ndef test_small():\n    \"examples\"\n    assert sliding_median([1, 3, -1, -3, 5, 3, 6, 7], 3) == [1, -1, -1, 3, 5, 6]\n    assert sliding_median([1, 2, 3, 4], 2) == [1.5, 2.5, 3.5] and sliding_median([5], 1) == [5] and sliding_median([1, 2], 3) == []\ndef test_random():\n    \"against sorting\"\n    rng = random.Random(4)\n    for _ in range(200):\n        xs = [rng.randint(-5, 5) for _ in range(rng.randint(0, 20))]\n        k = rng.randint(1, 6)\n        assert sliding_median(xs, k) == [statistics.median(xs[i:i + k]) for i in range(len(xs) - k + 1)], (xs, k)\ndef test_speed():\n    \"O(n log k)\"\n    rng = random.Random(5)\n    xs = [rng.randint(-10**6, 10**6) for _ in range(100_000)]\n    t0 = time.perf_counter()\n    out = sliding_median(xs, 1000)\n    assert len(out) == 99_001 and time.perf_counter() - t0 < 6")

## What an interview is like

A typical problem-solving interview lasts 45 to 60 minutes. After a few minutes of introductions, you're given one or two problems, usually in a shared editor or on a whiteboard, and asked to talk as you work. At the end there may be time for your questions.

The person interviewing you usually wants to see:

- **Understanding**: do you ask about the inputs, the sizes, the awkward cases, before diving in?
- **Problem solving**: can you find a correct approach, judge its cost, and improve it with reasons?
- **Code**: is it correct, readable, and organised into sensible pieces?
- **Testing**: do you check your own code with examples, including edge cases, without being asked?
- **Communication**: can someone follow your thinking while you work?

A working, well-explained O(n log n) solution with tests usually impresses more than an O(n) one that appears in silence and fails on an empty list. And interviews are a skill like any other: practice, out loud, makes a large difference. Rehearse with a friend, or record yourself talking through a problem.

## The method

Week 14 built problem solving on George Pólya's four steps (understand, plan, carry out, look back). In an interview they become seven, said out loud:

1. **Understand.** Restate the problem in your own words. Ask: how big can the input be? Can it be empty? Are there duplicates or negative numbers? What should happen on bad input? Is the input sorted?
2. **Examples.** Work one or two small examples by hand, including an awkward one. They often reveal a misunderstanding early.
3. **Brute force.** Describe the simplest correct approach and its cost, even if it's slow. It proves you understand, and gives you something to improve.
4. **Improve.** Ask what the brute force does repeatedly, and which structure or pattern would avoid it: sorting, a set or dict, two pointers, a window, prefix sums, a heap, a stack. Say the new cost before coding.
5. **Code.** Write clearly, with good names. If a piece is fiddly, give it a helper function and say what it does.
6. **Test.** Walk through your examples with the code, line by line. Then try edge cases: empty, one item, all equal, the largest values.
7. **Analyse.** State the time and space costs, and what would change with different constraints.

## Worked example: the longest run

Here's the method used on one problem, written as the conversation might go.

> **Problem**: given a list of integers, find the length of the longest run of **consecutive** values that can be formed from them, in any order. For `[100, 4, 200, 1, 3, 2]`, it's 4 (1, 2, 3, 4).

**Understand.** "So the numbers don't have to be next to each other in the list; I just need the longest set of values like 1, 2, 3, 4 that are all present. Can there be duplicates? Negative numbers? How long can the list be?" (Answer: duplicates and negatives are allowed, and the list can have a million items.) "And an empty list gives 0."

**Examples.** `[100, 4, 200, 1, 3, 2]` gives 4. `[0, 0, 1]` gives 2 (duplicates don't extend a run). `[]` gives 0. `[5]` gives 1.

**Brute force.** "Sort, then walk through counting runs, skipping duplicates. That's O(n log n) for the sort." That's already decent. "Without sorting: for each value, count upwards while the next value is present, using a set for lookups. But that's O(n²) when the values are one long run, because I'd count the same run from every starting point."

**Improve.** "The repeated work is starting a count in the middle of a run. If I only start counting from values whose predecessor **isn't** present, each run is counted once, from its start. Every value is then looked at a constant number of times: O(n) on average with a set."

**Code**, then **test** with the examples (out loud: "for 4, 3 is present, so skip; for 1, 0 isn't, so count 1, 2, 3, 4"), then **analyse**: "O(n) time on average, O(n) extra space for the set. The sorting version uses O(1) extra if I may sort in place, so it's a trade-off."

## Worked example: follow-ups

Interviews often continue with **follow-up questions** that change a constraint. They test whether you understand **why** your solution works, so you can adapt it. Common ones:

- **"What if the input is too big for memory?"** Stream it: one pass, keeping only a summary (a heap of size k, counts, a window). Or sort it externally and merge (Week 47 Day 5).
- **"What if it arrives one item at a time and you must answer after each?"** You need a structure that updates cheaply: a heap, a balanced tree, two heaps for medians, a dict of run boundaries.
- **"What if it's called many times on the same data?"** Do work once up front: sort, build prefix sums, a trie, an index.
- **"Can you do it in O(1) extra space?"** Two pointers, in-place swaps, or reusing the input.

For the longest run, "numbers arrive one at a time; report the longest run after each" has a neat answer: keep a dict from each run's **endpoints** to its length. A new value x joins the run ending at x − 1 and the run starting at x + 1, and only the new run's two ends need updating. Each arrival is O(1) on average. Today's exercises include it.

## When you're stuck

Everyone gets stuck sometimes, and how you handle it is part of what's being judged. Useful moves, said out loud:

- **Go back to examples.** Solve a small one by hand and watch what **you** do; the algorithm is often hiding there.
- **Simplify.** Solve an easier version (sorted input, no duplicates, k = 1), then generalise.
- **Run through the toolbox.** Would sorting help? A dict? Two pointers? A heap? Prefix sums? A stack? Breadth-first search?
- **Write the brute force.** A working slow solution beats a half-written fast one, and the slow one often shows the repeated work.
- **Take hints well.** If the interviewer suggests something, it's information, not failure: follow it and say what it changes.

And manage the clock: if you're ten minutes from the end, finish and test what you have rather than starting a better idea you can't complete. Say what you would do with more time.

## Practice

For each problem, before coding, write a comment block with steps 1 to 4: your clarifying assumptions, two examples, the brute force and its cost, and your improved idea and its cost. The checks only test the code, but the comments are the half of the interview that code can't show.

## Quiz

## Daily challenge

## Before you go

Pick one problem from this week and explain your solution out loud, start to finish, in under five minutes, as if to an interviewer. Record yourself if you can, and listen back: where did you go quiet? Tomorrow: the debrief of timed set 2.